# Visualização — Input_ML

Carrega e plota individualmente os arquivos `*_alinhado_ml.csv` da pasta `Input_ML`.

Altere `SUJEITO` (ex.: `"01"`, `"15"`, `"29"`) para escolher o arquivo.

In [399]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

INPUT_DIR = Path("Input_ML")
OUTPUT_DIR = Path("Inputs_DP")
SUJEITO = "20"
JANELA_MM_SEG = 1.0  # janela da média móvel para estimar e remover o drift (s)

ACC_COLS = ["accX_m_s2", "accY_m_s2", "accZ_m_s2"]
GYRO_COLS = ["gyroX_rad_s", "gyroY_rad_s", "gyroZ_rad_s"]
VEL_CORR_COL = "velocidade_corrigida_smart_m_s"
DESLOC_CORR_COL = "deslocamento_corrigido_smart_m"
VICON_COL = "vicon_esternoZ_cm"
TIME_COL = "Time"

In [400]:
def caminho_sujeito(sujeito: str) -> Path:
    return INPUT_DIR / f"{sujeito}_alinhado_ml.csv"


def listar_arquivos(input_dir: Path = INPUT_DIR) -> list[Path]:
    arquivos = sorted(input_dir.glob("*_alinhado_ml.csv"))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em {input_dir.resolve()}")
    return arquivos


def carregar_sujeito(sujeito: str) -> pd.DataFrame:
    path = caminho_sujeito(sujeito)
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    df = pd.read_csv(path)
    faltando = {TIME_COL, *ACC_COLS, *GYRO_COLS, VICON_COL} - set(df.columns)
    if faltando:
        raise ValueError(f"Colunas ausentes em {path.name}: {sorted(faltando)}")
    return df


def _layout_base(fig: go.Figure, sujeito: str, titulo: str, n_amostras: int, duracao_s: float) -> go.Figure:
    fig.update_layout(
        title=f"Sujeito {sujeito} — {titulo} ({n_amostras} amostras, {duracao_s:.1f} s)",
        template="plotly_dark",
        width=1200,
        plot_bgcolor="black",
        paper_bgcolor="black",
        font=dict(color="white"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    return fig


def plotar_sinais(sujeito: str = SUJEITO) -> go.Figure:
    """Plota acelerômetro, giroscópio e Vicon."""
    df = carregar_sujeito(sujeito)
    t = df[TIME_COL]

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        subplot_titles=(
            "Acelerômetro (m/s²)",
            "Giroscópio (rad/s)",
            "Vicon (cm)",
        ),
    )

    cores_acc = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    for col, cor in zip(ACC_COLS, cores_acc):
        fig.add_trace(
            go.Scatter(x=t, y=df[col], mode="lines", name=col, line=dict(color=cor)),
            row=1, col=1,
        )

    cores_gyro = ["#9467bd", "#8c564b", "#e377c2"]
    for col, cor in zip(GYRO_COLS, cores_gyro):
        fig.add_trace(
            go.Scatter(x=t, y=df[col], mode="lines", name=col, line=dict(color=cor)),
            row=2, col=1,
        )

    fig.add_trace(
        go.Scatter(
            x=t, y=df[VICON_COL], mode="lines", name=VICON_COL,
            line=dict(color="#00d4aa", width=2),
        ),
        row=3, col=1,
    )

    fig = _layout_base(fig, sujeito, "sinais brutos", len(df), float(t.iloc[-1]))
    fig.update_layout(height=900)
    fig.update_xaxes(title_text="Tempo (s)", row=3, col=1)

    return fig


def calcular_aceleracao_resultante_yz(df: pd.DataFrame) -> pd.Series:
    """Magnitude da aceleração no plano Y–Z: sqrt(accY² + accZ²)."""
    acc_y = df["accY_m_s2"].to_numpy(dtype=float)
    acc_z = df["accZ_m_s2"].to_numpy(dtype=float)
    return pd.Series(np.hypot(acc_y, acc_z), index=df.index, name="acc_resultante_yz_m_s2")


def plotar_aceleracao_resultante_yz(sujeito: str = SUJEITO) -> go.Figure:
    """Plota a aceleração resultante entre os eixos Y e Z."""
    df = carregar_sujeito(sujeito)
    t = df[TIME_COL]
    acc_resultante = calcular_aceleracao_resultante_yz(df)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=t, y=acc_resultante, mode="lines", name="acc_resultante_yz",
            line=dict(color="#ffd166", width=2),
        )
    )

    fig = _layout_base(
        fig, sujeito, "aceleração resultante Y–Z", len(df), float(t.iloc[-1])
    )
    fig.update_layout(height=400)
    fig.update_xaxes(title_text="Tempo (s)")
    fig.update_yaxes(title_text="m/s²")

    return fig


def _compute_dt(time_values: np.ndarray) -> np.ndarray:
    dt = np.diff(time_values, prepend=time_values[0])
    if len(dt) > 1:
        positive_dt = dt[dt > 0]
        fallback = float(np.median(positive_dt)) if len(positive_dt) else 1.0 / 60.0
    else:
        fallback = 1.0 / 60.0
    return np.where(dt <= 0, fallback, dt)


def corrigir_drift_media_movel(
    sinal: pd.Series, janela_seg: float, dt_medio: float
) -> tuple[pd.Series, pd.Series]:
    """Estima o drift com média móvel e subtrai do sinal."""
    janela = max(3, int(round(janela_seg / dt_medio)))
    drift = sinal.rolling(window=janela, center=True, min_periods=1).mean()
    sinal_corrigido = sinal - drift
    return drift, sinal_corrigido


def calcular_velocidade_acc_y(
    df: pd.DataFrame, janela_mm_seg: float = JANELA_MM_SEG
) -> tuple[pd.Series, pd.Series, pd.Series]:
    """Integra accY e corrige drift da velocidade com média móvel."""
    t = df[TIME_COL].to_numpy(dtype=float)
    dt = _compute_dt(t)
    dt_medio = float(np.median(dt))

    acc_y = df["accY_m_s2"].to_numpy(dtype=float)
    acc_y_centered = acc_y - np.nanmedian(acc_y)
    velocidade = pd.Series(
        np.cumsum(acc_y_centered * dt), index=df.index, name="vel_accY_m_s"
    )

    drift, velocidade_corrigida = corrigir_drift_media_movel(
        velocidade, janela_mm_seg, dt_medio
    )
    drift.name = "drift_media_movel_m_s"
    velocidade_corrigida.name = "vel_accY_corrigida_m_s"
    return velocidade, drift, velocidade_corrigida


def plotar_velocidade_acc_y(
    sujeito: str = SUJEITO, janela_mm_seg: float = JANELA_MM_SEG
) -> go.Figure:
    """Plota velocidade bruta, drift estimado e velocidade corrigida."""
    df = carregar_sujeito(sujeito)
    t = df[TIME_COL]
    velocidade, drift, velocidade_corrigida = calcular_velocidade_acc_y(df, janela_mm_seg)

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=t, y=velocidade, mode="lines", name="velocidade bruta",
            line=dict(color="#6c757d", width=1, dash="dot"),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t, y=drift, mode="lines", name=f"drift (MM {janela_mm_seg:.1f} s)",
            line=dict(color="#ff9f1c", width=1.5),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t, y=velocidade_corrigida, mode="lines", name="velocidade corrigida",
            line=dict(color="#4cc9f0", width=2),
        )
    )

    fig = _layout_base(
        fig,
        sujeito,
        f"velocidade accY — drift corrigido (MM {janela_mm_seg:.1f} s)",
        len(df),
        float(t.iloc[-1]),
    )
    fig.update_layout(height=450)
    fig.update_xaxes(title_text="Tempo (s)")
    fig.update_yaxes(title_text="m/s")

    return fig


def calcular_deslocamento_acc_y(
    df: pd.DataFrame, janela_mm_seg: float = JANELA_MM_SEG
) -> tuple[pd.Series, pd.Series, pd.Series]:
    """2ª integral da velocidade corrigida, com correção de drift no deslocamento."""
    t = df[TIME_COL].to_numpy(dtype=float)
    dt = _compute_dt(t)
    dt_medio = float(np.median(dt))
    _, _, velocidade_corrigida = calcular_velocidade_acc_y(df, janela_mm_seg)

    deslocamento = pd.Series(
        np.cumsum(velocidade_corrigida.to_numpy(dtype=float) * dt),
        index=df.index,
        name="desloc_accY_m",
    )
    drift, deslocamento_corrigido = corrigir_drift_media_movel(
        deslocamento, janela_mm_seg, dt_medio
    )
    drift.name = "drift_desloc_mm_m"
    deslocamento_corrigido.name = "desloc_accY_corrigido_m"
    return deslocamento, drift, deslocamento_corrigido


def plotar_deslocamento_acc_y(
    sujeito: str = SUJEITO, janela_mm_seg: float = JANELA_MM_SEG
) -> go.Figure:
    """Plota deslocamento bruto, drift estimado e deslocamento corrigido."""
    df = carregar_sujeito(sujeito)
    t = df[TIME_COL]
    deslocamento, drift, deslocamento_corrigido = calcular_deslocamento_acc_y(
        df, janela_mm_seg
    )

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=t, y=deslocamento, mode="lines", name="deslocamento bruto",
            line=dict(color="#6c757d", width=1, dash="dot"),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t, y=drift, mode="lines", name=f"drift desloc. (MM {janela_mm_seg:.1f} s)",
            line=dict(color="#ff9f1c", width=1.5),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t, y=deslocamento_corrigido, mode="lines", name="deslocamento corrigido",
            line=dict(color="#f72585", width=2),
        )
    )

    fig = _layout_base(
        fig,
        sujeito,
        f"deslocamento accY — 2ª integral + drift corrigido (MM {janela_mm_seg:.1f} s)",
        len(df),
        float(t.iloc[-1]),
    )
    fig.update_layout(height=450)
    fig.update_xaxes(title_text="Tempo (s)")
    fig.update_yaxes(title_text="m")

    return fig


def plotar_deslocamento_com_vicon(
    sujeito: str = SUJEITO, janela_mm_seg: float = JANELA_MM_SEG
) -> go.Figure:
    """Plota deslocamento integrado corrigido (accY) e Vicon centrado."""
    df = carregar_sujeito(sujeito)
    t = df[TIME_COL]
    _, _, deslocamento_corrigido = calcular_deslocamento_acc_y(df, janela_mm_seg)
    deslocamento_cm = deslocamento_corrigido * 100.0
    vicon_cm = df[VICON_COL] - df[VICON_COL].median()

    fig = go.Figure()
    fig.add_trace(
        go.Scatter(
            x=t, y=deslocamento_cm, mode="lines", name="deslocamento accY (corrigido)",
            line=dict(color="#f72585", width=2),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=t, y=vicon_cm, mode="lines", name="Vicon (centrado)",
            line=dict(color="#00d4aa", width=2, dash="dot"),
        )
    )

    fig = _layout_base(
        fig,
        sujeito,
        f"deslocamento accY vs Vicon centrado (MM {janela_mm_seg:.1f} s)",
        len(df),
        float(t.iloc[-1]),
    )
    fig.update_layout(height=450)
    fig.update_xaxes(title_text="Tempo (s)")
    fig.update_yaxes(title_text="cm")

    return fig


def montar_dataframe_inputs_dp(
    df: pd.DataFrame, janela_mm_seg: float = JANELA_MM_SEG
) -> pd.DataFrame:
    """Monta tabela igual ao Input_ML + velocidade e deslocamento corrigidos."""
    _, _, velocidade_corrigida = calcular_velocidade_acc_y(df, janela_mm_seg)
    _, _, deslocamento_corrigido = calcular_deslocamento_acc_y(df, janela_mm_seg)

    return pd.DataFrame(
        {
            TIME_COL: df[TIME_COL],
            ACC_COLS[0]: df[ACC_COLS[0]],
            ACC_COLS[1]: df[ACC_COLS[1]],
            ACC_COLS[2]: df[ACC_COLS[2]],
            GYRO_COLS[0]: df[GYRO_COLS[0]],
            GYRO_COLS[1]: df[GYRO_COLS[1]],
            GYRO_COLS[2]: df[GYRO_COLS[2]],
            VEL_CORR_COL: velocidade_corrigida,
            DESLOC_CORR_COL: deslocamento_corrigido,
            VICON_COL: df[VICON_COL],
        }
    )


def salvar_inputs_dp(
    sujeito: str = SUJEITO,
    janela_mm_seg: float = JANELA_MM_SEG,
    output_dir: Path = OUTPUT_DIR,
) -> Path:
    """Salva um sujeito em Inputs_DP com as colunas derivadas."""
    output_dir.mkdir(parents=True, exist_ok=True)
    df = carregar_sujeito(sujeito)
    export_df = montar_dataframe_inputs_dp(df, janela_mm_seg)
    path = output_dir / f"{sujeito}_alinhado_ml.csv"
    export_df.to_csv(path, index=False)
    return path

In [401]:
print("Sujeitos disponíveis:")
for arq in listar_arquivos():
    print(f"  {arq.stem.replace('_alinhado_ml', '')}")

Sujeitos disponíveis:
  01
  02
  03
  04
  06
  08
  09
  10
  11
  12
  13
  14
  15
  16
  17
  18
  19
  20
  21
  22
  23
  24
  25
  26
  27
  28
  29


In [402]:
plotar_sinais(SUJEITO)

In [403]:
plotar_aceleracao_resultante_yz(SUJEITO)

In [404]:
plotar_velocidade_acc_y(SUJEITO)

In [405]:
plotar_deslocamento_acc_y(SUJEITO)

In [406]:
plotar_deslocamento_com_vicon(SUJEITO)

In [407]:
path_salvo = salvar_inputs_dp(SUJEITO)
print(f"Salvo: {path_salvo.resolve()}")

Salvo: /Users/Rodacki/Desktop/Hoffmann/UTT/Inputs_DP/20_alinhado_ml.csv
